In [3]:
# Robust loader + full cleaning pipeline for Zomato dataset
import pandas as pd
import numpy as np
import requests
from io import StringIO

urls = [
    "https://raw.githubusercontent.com/SowmyaaKarthik/Zomato--Data-Analysis-Data-Visualization/main/zomato.csv",
    "https://raw.githubusercontent.com/SouGuit/Zomato_Dataset_Analysis/main/zomato.csv",
    "https://raw.githubusercontent.com/chiragsamal/Zomato/main/zomato.csv",
    # add more raw URLs here if you find them
]

def load_first_working(urls, timeout=10):
    for url in urls:
        try:
            print(f"Trying: {url}")
            r = requests.get(url, timeout=timeout)
            r.raise_for_status()
            # some CSVs may include weird encodings; try utf-8 then latin-1
            try:
                return pd.read_csv(StringIO(r.text))
            except Exception:
                return pd.read_csv(StringIO(r.content.decode('latin-1')))
        except Exception as e:
            print(f"Failed: {url} -> {e}")
    return None

df = load_first_working(urls)

if df is None:
    raise RuntimeError(
        "Could not fetch any of the provided URLs. "
        "Options:\n"
        " 1) Download dataset from Kaggle (https://www.kaggle.com/datasets/shrutimehta/zomato-restaurants-data) and upload to this session,\n"
        " 2) Paste a working raw .csv GitHub URL here."
    )

print("Loaded shape:", df.shape)
print(df.columns.tolist()[:30])

# ---------- Cleaning (same steps as before, with some safety) ----------
# 1) drop columns with >50% missing
df = df.copy()
df = df.dropna(axis=1, thresh=len(df)*0.5)

# 2) sensible fills for common columns (guarded with if-exists)
for col, fill in [('Cuisines','Not Specified'), ('City','Unknown'), ('Locality','Unknown')]:
    if col in df.columns:
        df[col] = df[col].fillna(fill)

# 3) remove exact duplicate rows
df = df.drop_duplicates().reset_index(drop=True)

# 4) standardize text columns (if exist)
text_cols = [c for c in ['Restaurant Name','Restaurant Name','City','Locality','Cuisines'] if c in df.columns]
for col in text_cols:
    df[col] = df[col].astype(str).str.strip().replace({'nan':''}).str.title()

# 5) numeric casting for likely numeric cols (safe)
num_map = {
    'Votes': 'Votes',
    'Average Cost for two': 'Average Cost for two',
    'Average Cost for two (₹)': 'Average Cost for two',
    'Aggregate rating': 'Aggregate rating',
    'Rate': 'Rate'
}
for k in num_map:
    if k in df.columns:
        df[k] = pd.to_numeric(df[k].astype(str).str.replace('[^0-9.\-]', '' , regex=True), errors='coerce')

# 6) outlier handling via IQR on existing numeric columns
num_cols = [c for c in df.select_dtypes(include=[np.number]).columns.tolist() if c in ['Votes','Average Cost for two','Aggregate rating']]
for col in num_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    before = len(df)
    df = df[(df[col].isna()) | ((df[col] >= lower) & (df[col] <= upper))]  # keep NaNs for later imputation
    after = len(df)
    print(f"{col}: removed {before-after} rows as outliers")

# 7) final quick fixes: drop rows with no restaurant name
if 'Restaurant Name' in df.columns:
    df = df[df['Restaurant Name'].str.strip().astype(bool)]

print("Cleaned shape:", df.shape)

# 8) save cleaned file locally
out_path = "Cleaned_Zomato_Dataset.csv"
df.to_csv(out_path, index=False)
print("Saved cleaned dataset to:", out_path)



Trying: https://raw.githubusercontent.com/SowmyaaKarthik/Zomato--Data-Analysis-Data-Visualization/main/zomato.csv
Loaded shape: (9551, 21)
['Restaurant ID', 'Restaurant Name', 'Country Code', 'City', 'Address', 'Locality', 'Locality Verbose', 'Longitude', 'Latitude', 'Cuisines', 'Average Cost for two', 'Currency', 'Has Table booking', 'Has Online delivery', 'Is delivering now', 'Switch to order menu', 'Price range', 'Aggregate rating', 'Rating color', 'Rating text', 'Votes']
Average Cost for two: removed 853 rows as outliers
Aggregate rating: removed 2130 rows as outliers
Votes: removed 728 rows as outliers
Cleaned shape: (5840, 21)
Saved cleaned dataset to: Cleaned_Zomato_Dataset.csv
